In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from module_1 import read_and_clean
from module_2 import compute_ecg_features, compute_resp_features_
import pickle

%matplotlib qt

In [2]:
fs=250
subject = 4
config = "wire"
activity = "walking"

dev_path = f"subject_{subject}/{config}/dev/{activity}_dev.bin"
ref_ecg_path = f"subject_{subject}/{config}/reference/{activity}_ecg.EDF"
ref_resp_path = f"subject_{subject}/{config}/reference/{activity}_resp.acq"

In [ ]:
# data = read_and_clean(dev_path, ref_ecg_path, ref_resp_path)

In [ ]:
# ── load ──
with open("all_device_data.pkl", "rb") as f:
    data = pickle.load(f)

# everything comes back exactly as it was
signal = data["subject_4/wire/walking"]["respiration"]["prep_ip"]

In [4]:

'''
Accessing data class: 

data.respiration.raw_ip       # raw
data.respiration.prep_ip      # preprocessed
data.respiration.dev_ip       # aligned

data.ecg.raw_dev_lead2        # raw
data.ecg.prep_lead2           # preprocessed
data.ecg.dev_lead2            # aligned
'''

'\nAccessing data class: \n\ndata.respiration.raw_ip       # raw\ndata.respiration.prep_ip      # preprocessed\ndata.respiration.dev_ip       # aligned\n\ndata.ecg.raw_dev_lead2        # raw\ndata.ecg.prep_lead2           # preprocessed\ndata.ecg.dev_lead2            # aligned\n'

## below are the raw signals

In [5]:
raw_dev_lead2 = np.array(data.ecg.raw_dev_lead2)
raw_ref_lead2 = np.array(data.ecg.raw_ref_lead2)

raw_ip = np.array(data.respiration.raw_ip)
raw_gyr = np.array(data.respiration.raw_gyr)
raw_ref = np.array(data.respiration.raw_ref)

In [6]:
plt.figure()
plt.plot(raw_dev_lead2, label="raw_dev_lead2")
plt.plot(raw_ref_lead2, label="raw_ref_lead2")
plt.title("raw_lead2_comparison")
plt.legend()
plt.show

plt.figure()
plt.plot(raw_ip, label="raw_ip")
plt.plot(raw_gyr, label="raw_gyr")
plt.plot(raw_ref, label="raw_ref")
plt.title("raw_respiration_comparison")
plt.legend()
plt.show

<function matplotlib.pyplot.show(*, block=None)>

## below are the pre aligned preprocessed signals

In [7]:
pre_aligned_prep_ip = np.array(data.respiration.pre_aligned_prep_dev_ip)
pre_aligned_prep_gyr = np.array(data.respiration.pre_aligned_prep_dev_gyr)
pre_aligned_prep_ref_resp = np.array(data.respiration.pre_aligned_prep_ref_respiration)

In [8]:
plt.figure()
plt.plot(pre_aligned_prep_ip, label="pre_aligned_prep_ip")
plt.plot(pre_aligned_prep_gyr, label="pre_aligned_prep_gyr")
plt.plot(pre_aligned_prep_ref_resp, label="pre_aligned_prep_ref_resp")
plt.title("pre_aligned_preprocessed_respiration_comparison")
plt.legend()
plt.show

<function matplotlib.pyplot.show(*, block=None)>

## below are the filtered signals

In [9]:
prep_lead2 = np.array(data.ecg.prep_lead2)
prep_ref_ecg = np.array(data.ecg.prep_ref_ecg)

prep_ip = np.array(data.respiration.prep_ip)
prep_gyr = np.array(data.respiration.prep_gyr)
prep_ref_resp = np.array(data.respiration.prep_ref)

In [10]:
plt.figure()
plt.plot(prep_lead2, label="prep_lead2")
plt.plot(prep_ref_ecg, label="prep_ref_ecg")
plt.title("filtered_lead2_comparison")
plt.legend()
plt.show

plt.figure()
plt.plot(prep_ip, label="prep_ip")
plt.plot(prep_gyr, label="prep_gyr")
plt.plot(prep_ref_resp, label="prep_ref_resp")
plt.title("filtered_respiration_comparison")
plt.legend()
plt.show

<function matplotlib.pyplot.show(*, block=None)>

## below are the aligned signals

In [11]:
dev_lead2 = np.array(data.ecg.dev_lead2)
ref_lead2 = np.array(data.ecg.ref_lead2)

dev_ip = np.array(data.respiration.dev_ip)
dev_gyr = np.array(data.respiration.dev_gyr)
ref_respiration = np.array(data.respiration.ref_respiration)

In [12]:
plt.figure()
plt.plot(dev_lead2, label="aligned_lead2")
plt.plot(ref_lead2, label="aligned_ref_lead2")
plt.title("aligned_lead2_comparison")
plt.legend()
plt.show

plt.figure()
plt.plot(dev_ip, label="aligned_ip")
plt.plot(dev_gyr, label="aligned_gyr")
plt.plot(ref_respiration, label="aligned_ref_resp")
plt.title("aligned_respiration_comparison")
plt.legend()
plt.show

<function matplotlib.pyplot.show(*, block=None)>

# Respiration segmentation and processing

In [13]:
resp_sig_len = len(ref_respiration)
resp_window_sec = 30
resp_step_sec=10

resp_W = int(resp_window_sec * fs)
resp_step = int(resp_step_sec * fs)
resp_segments = [(s, s + resp_W) for s in range(0, resp_sig_len - resp_W + 1, resp_step)]

In [14]:
final_rr = []
resp_ref_all_segs = []

spi_ip_all_segs = []
spi_gyr_all_segs = []
spi_ref_all_segs = []

for i, (s, e) in enumerate(resp_segments):
    resp_ip, spi_ip = compute_resp_features_(dev_ip[s:e])
    resp_gyr, spi_gyr = compute_resp_features_(dev_gyr[s:e])
    resp_ref, spi_ref = compute_resp_features_(ref_respiration[s:e])
    
    resp_ref_all_segs.append(resp_ref)
    
    spi_ip_all_segs.append(spi_ip)
    spi_gyr_all_segs.append(spi_gyr)
    spi_ref_all_segs.append(spi_ref)

    ip_nan = np.isnan(resp_ip)
    gyr_nan = np.isnan(resp_gyr)

    if ip_nan and gyr_nan:
        final_rr.append(float('nan'))
    elif ip_nan or gyr_nan:
        final_rr.append(float(resp_gyr if ip_nan else resp_ip))
    else:
        final_rr.append(float(np.mean([resp_ip, resp_gyr])))

In [15]:
resp_error = []
for idx, rr in enumerate(final_rr):
    resp_error.append(rr - resp_ref_all_segs[idx])

In [16]:
rr_comp = pd.DataFrame({
    'dev_rr': final_rr,
    'ref_rr': resp_ref_all_segs,
    'error': resp_error
})
rr_comp

,dev_rr,ref_rr,error
0,20.325512,17.164999,3.160513
1,20.807204,20.864670,-0.057465
2,22.163893,23.196195,-1.032302
3,22.066282,21.149284,0.916998
4,22.541492,20.388135,2.153357
5,22.655320,21.350161,1.305160
6,22.705514,23.522953,-0.817439
7,21.725744,21.420068,0.305675
8,18.329537,18.512872,-0.183335
9,19.095322,19.353820,-0.258499


In [17]:
spi_comp = pd.DataFrame({
    'spi_ip': spi_ip_all_segs,
    'spi_gyr': spi_gyr_all_segs,
    'spi_ref': spi_ref_all_segs
})
spi_comp

,spi_ip,spi_gyr,spi_ref
0,0.601820,0.726050,0.744212
1,0.621408,0.770597,0.835477
2,0.603765,0.739801,0.834115
3,0.581913,0.734954,0.665812
4,0.599016,0.747202,0.657333
5,0.734814,0.739901,0.814937
6,0.747316,0.766155,0.824193
7,0.638404,0.687972,0.750280
8,0.485561,0.640718,0.625360
9,0.391648,0.683387,0.624181


In [18]:
resp_sig_len = len(pre_aligned_prep_ref_resp)
resp_window_sec = 30
resp_step_sec=10

resp_W = int(resp_window_sec * fs)
resp_step = int(resp_step_sec * fs)
resp_segments = [(s, s + resp_W) for s in range(0, resp_sig_len - resp_W + 1, resp_step)]

In [19]:
final_rr = []
resp_ref_all_segs = []

spi_ip_all_segs = []
spi_gyr_all_segs = []
spi_ref_all_segs = []

for i, (s, e) in enumerate(resp_segments):
    resp_ip, spi_ip = compute_resp_features_(pre_aligned_prep_ip[s:e])
    resp_gyr, spi_gyr = compute_resp_features_(pre_aligned_prep_gyr[s:e])
    resp_ref, spi_ref = compute_resp_features_(pre_aligned_prep_ref_resp[s:e])
    
    resp_ref_all_segs.append(resp_ref)
    
    spi_ip_all_segs.append(spi_ip)
    spi_gyr_all_segs.append(spi_gyr)
    spi_ref_all_segs.append(spi_ref)

    ip_nan = np.isnan(resp_ip)
    gyr_nan = np.isnan(resp_gyr)

    if ip_nan and gyr_nan:
        final_rr.append(float('nan'))
    elif ip_nan or gyr_nan:
        final_rr.append(float(resp_gyr if ip_nan else resp_ip))
    else:
        final_rr.append(float(np.mean([resp_ip, resp_gyr])))

In [20]:
resp_error = []
for idx, rr in enumerate(final_rr):
    resp_error.append(rr - resp_ref_all_segs[idx])

In [21]:
rr_comp = pd.DataFrame({
    'dev_rr': final_rr,
    'ref_rr': resp_ref_all_segs,
    'error': resp_error
})
rr_comp

,dev_rr,ref_rr,error
0,19.056717,17.164999,1.891719
1,20.814313,20.864670,-0.050357
2,22.163893,23.196195,-1.032302
3,22.066282,21.149284,0.916998
4,22.541492,20.388135,2.153357
5,22.655320,21.350161,1.305160
6,22.705514,23.522953,-0.817439
7,21.725744,21.420068,0.305675
8,18.329537,18.512872,-0.183335
9,19.095322,19.353820,-0.258499


# ECG Segmentation and Processing

In [22]:
ecg_sig_len = len(ref_lead2)
ecg_window_sec = 10

ecg_W    = int(ecg_window_sec * fs)
ecg_step = ecg_W
n = ecg_sig_len // ecg_W

ecg_segments = [(i * ecg_step, i * ecg_step + ecg_W) for i in range(n)]

In [23]:
dev_snr_all_segs = []
dev_hr_all_segs = []
dev_rmssd_all_segs = []

ref_snr_all_segs = []
ref_hr_all_segs = []
ref_rmssd_all_segs = []

for i, (s, e) in enumerate(ecg_segments):
    dev_snr, dev_hr, dev_rmssd = compute_ecg_features(dev_lead2)
    ref_snr, ref_hr, ref_rmssd = compute_ecg_features(ref_lead2)

    dev_snr_all_segs.append(dev_snr)
    dev_hr_all_segs.append(dev_hr)
    dev_rmssd_all_segs.append(dev_rmssd)

    ref_snr_all_segs.append(ref_snr)
    ref_hr_all_segs.append(ref_hr)
    ref_rmssd_all_segs.append(ref_rmssd)

In [24]:
ecg_error = []
for idx, hr in enumerate(dev_hr_all_segs):
    ecg_error.append(hr - ref_hr_all_segs[idx])

In [25]:
hr_comp = pd.DataFrame({
    'dev_hr': dev_hr_all_segs,
    'ref_hr': ref_hr_all_segs,
    'error': ecg_error
})
hr_comp

,dev_hr,ref_hr,error
0,89.088824,88.992011,0.096813
1,89.088824,88.992011,0.096813
2,89.088824,88.992011,0.096813
3,89.088824,88.992011,0.096813
4,89.088824,88.992011,0.096813
5,89.088824,88.992011,0.096813
6,89.088824,88.992011,0.096813
7,89.088824,88.992011,0.096813
8,89.088824,88.992011,0.096813
9,89.088824,88.992011,0.096813
